In [2]:
from sklearn.metrics import *
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from efficient_apriori import apriori
import warnings
import numba

warnings.filterwarnings('ignore')

In [3]:
# Carregar os dados
aisles_df = pd.read_csv("./data_mba/aisles.csv")
orders_df = pd.read_csv("./data_mba/orders.csv")
departments_df = pd.read_csv("./data_mba/departments.csv")
products_df = pd.read_csv("./data_mba/products.csv")
order_products_prior_df = pd.read_csv("./data_mba/order_products__prior.csv")
order_products_train_df = pd.read_csv("./data_mba/order_products__train.csv")

# Criar lista de dataframes para perfil
dataframes = {
    "aisles": aisles_df,
    "orders": orders_df,
    "departments": departments_df,
    "products": products_df,
    "order_products_prior": order_products_prior_df,
    "order_products_train": order_products_train_df
}
display(orders_df.head(), products_df.head(), aisles_df.head(), departments_df.head(), order_products_prior_df.head(), order_products_train_df.head())

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


In [4]:
for df, name in zip([orders_df, products_df, aisles_df, departments_df, order_products_prior_df, order_products_train_df],
                    ['orders', 'products', 'aisles', 'departments', 'order_products_prior', 'order_products_train']):
    print(f"Missing values in {name}:\n{df.isnull().sum()}\n")

Missing values in orders:
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

Missing values in products:
product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

Missing values in aisles:
aisle_id    0
aisle       0
dtype: int64

Missing values in departments:
department_id    0
department       0
dtype: int64

Missing values in order_products_prior:
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

Missing values in order_products_train:
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64



In [5]:
order_products = pd.concat([order_products_prior_df, order_products_train_df])

transactions = order_products.groupby("order_id")["product_id"].apply(list).reset_index()

transactions


,order_id,product_id
0,1,"[49302, 11109, 10246, 49683, 43633, 13176, 472..."
1,2,"[33120, 28985, 9327, 45918, 30035, 17794, 4014..."
2,3,"[33754, 24838, 17704, 21903, 17668, 46667, 174..."
3,4,"[46842, 26434, 39758, 27761, 10054, 21351, 225..."
4,5,"[13176, 15005, 47329, 27966, 23909, 48370, 132..."
...,...,...
3346078,3421079,[30136]
3346079,3421080,"[27845, 4932, 18811, 41950, 31717, 12935, 2512..."
3346080,3421081,"[38185, 12218, 32299, 3060, 20539, 35221, 12861]"
3346081,3421082,"[17279, 12738, 16797, 43352, 32700, 12023, 47941]"


In [6]:
print(order_products_prior_df.memory_usage(deep=True).sum() / 1e6, "MB")
print(orders_df.memory_usage(deep=True).sum() / 1e6, "MB")
print(products_df.memory_usage(deep=True).sum() / 1e6, "MB")
print(aisles_df.memory_usage(deep=True).sum() / 1e6, "MB")
print(departments_df.memory_usage(deep=True).sum() / 1e6, "MB")

1037.90378 MB
348.875598 MB
5.168367 MB
0.009796 MB
0.001499 MB


In [ ]:
import pandas as pd

chunk_size = 100000
merged_chunks = []

for chunk in pd.read_csv("./data_mba/order_products__prior.csv", chunksize=chunk_size):
    chunk = chunk.merge(orders_df[['order_id', 'user_id', 'order_dow', 'order_hour_of_day']], on='order_id', how='left')
    chunk = chunk.merge(products_df[['product_id', 'product_name', 'aisle_id', 'department_id']], on='product_id', how='left')
    chunk = chunk.merge(aisles_df[['aisle_id', 'aisle']], on='aisle_id', how='left')
    chunk = chunk.merge(departments_df[['department_id', 'department']], on='department_id', how='left')

    merged_chunks.append(chunk)

order_products_prior = pd.concat(merged_chunks, ignore_index=True)


: 

In [ ]:
order_products_prior.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_dow,order_hour_of_day,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,5,9,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,5,9,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,5,9,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,5,9,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,5,9,Natural Sweetener,17,13,baking ingredients,pantry
